# Leading Indicator — OSHA Form 300A analysis

**Question:** which kinds of workplaces have the highest recordable-injury rates, and how
different is that ranking from a ranking by raw injury count?

**Data:** OSHA Injury Tracking Application, Form 300A summary data, calendar year 2025,
submissions received 1 Jan – 15 Mar 2026 (`ITA_300A_Summary_Data_2025_through_03-15-2026_v2.csv`,
downloaded from <https://www.osha.gov/itadata>). One row per establishment per year.

---

## Section 1 — Setup and load

Milestone 1 loads the raw file and audits it. **No row is dropped and no value is altered in
this milestone** — M1 counts problems, M2 fixes them, and only after the exclusion list has
been decided by a human.

In [1]:
# Imports, repo-root anchoring, DuckDB connection, and the SQL runner.
#
# Path anchoring: a notebook has no __file__, and the working directory differs between
# Jupyter (starts in notebooks/) and `nbconvert` (starts wherever it was invoked). So we
# walk up from the current directory until we find the repo markers. Every path below is
# built from REPO_ROOT, never relative to the cwd.

from pathlib import Path
import duckdb
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk upward until a directory contains both CLAUDE.md and sql/."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists() and (candidate / "sql").is_dir():
            return candidate
    raise RuntimeError(f"repo root not found above {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SQL_DIR = REPO_ROOT / "sql"
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
CSV_PATH = DATA_DIR / "ITA_300A_Summary_Data_2025_through_03-15-2026_v2.csv"
DB_PATH = DATA_DIR / "leading_indicator.duckdb"

# Persistent database file, so the loaded table survives a kernel restart.
con = duckdb.connect(str(DB_PATH))


def run(filename: str, **params) -> pd.DataFrame:
    """Read one .sql file from sql/ and execute it, returning a DataFrame.

    {CSV_PATH} in the file is replaced with the absolute path to the raw CSV, which keeps
    the .sql files free of machine-specific paths. Extra keyword arguments substitute
    further {PLACEHOLDER} tokens.
    """
    sql = (SQL_DIR / filename).read_text()
    sql = sql.replace("{CSV_PATH}", str(CSV_PATH))
    for key, value in params.items():
        sql = sql.replace("{" + key + "}", str(value))
    result = con.sql(sql)
    # DDL (CREATE TABLE ...) returns no result set; hand back an empty frame.
    return result.df() if result is not None else pd.DataFrame()


# Show full tables rather than pandas' abbreviated view — these outputs are the evidence.
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("REPO_ROOT :", REPO_ROOT)
print("CSV_PATH  :", CSV_PATH)
print("CSV exists:", CSV_PATH.exists(),
      f"({CSV_PATH.stat().st_size / 1048576:.2f} MB)" if CSV_PATH.exists() else "")
print("DB_PATH   :", DB_PATH)
print("duckdb    :", duckdb.__version__, "| pandas:", pd.__version__)

REPO_ROOT : /Users/ashvijain/Leading Indicator
CSV_PATH  : /Users/ashvijain/Leading Indicator/data/ITA_300A_Summary_Data_2025_through_03-15-2026_v2.csv
CSV exists: True (80.68 MB)
DB_PATH   : /Users/ashvijain/Leading Indicator/data/leading_indicator.duckdb
duckdb    : 1.4.5 | pandas: 2.3.3


In [2]:
# Load the raw CSV into DuckDB as `raw_300a` (sql/01_load.sql).
#
# `sample_size = -1` makes DuckDB scan the entire file before choosing column types, so a
# text value 300,000 rows in cannot break a type inferred from the first few thousand rows.
# There is deliberately no `ignore_errors`: a row that will not parse must raise an error
# here rather than disappear silently.

import time

t0 = time.time()
run("01_load.sql")          # DDL — returns no result set
print(f"01_load.sql completed in {time.time() - t0:.1f}s")

n_raw = con.sql("SELECT COUNT(*) AS n FROM raw_300a").fetchone()[0]
print(f"raw_300a row count      : {n_raw:,}")
print(f"CSV data lines expected : 383,283  (383,284 lines incl. header)")
print(f"match                   : {n_raw == 383_283}")

01_load.sql completed in 2.7s
raw_300a row count      : 383,283
CSV data lines expected : 383,283  (383,284 lines incl. header)
match                   : True


In [3]:
# DESCRIBE the loaded table and compare it, name by name and in order, against the column
# list in PRD.md section 4 (taken from OSHA's published data dictionary). Any difference in
# name, order, or count is named explicitly rather than assumed away.

described = con.sql("DESCRIBE raw_300a").df()
display(described[["column_name", "column_type", "null"]])

prd_columns = """id establishment_name establishment_id ein company_name street_address city
state zip_code naics_code naics_year industry_description establishment_type size
annual_average_employees total_hours_worked no_injuries_illnesses total_deaths
total_dafw_cases total_djtr_cases total_other_cases total_dafw_days total_djtr_days
total_injuries total_skin_disorders total_respiratory_conditions total_poisonings
total_hearing_loss total_other_illnesses created_timestamp change_reason
year_filing_for""".split()

actual_columns = described["column_name"].tolist()

print(f"\nPRD section 4 columns : {len(prd_columns)}")
print(f"raw_300a columns      : {len(actual_columns)}")
print(f"identical, same order : {actual_columns == prd_columns}")
print(f"in PRD but not loaded : {[c for c in prd_columns if c not in actual_columns] or 'none'}")
print(f"loaded but not in PRD : {[c for c in actual_columns if c not in prd_columns] or 'none'}")

# Type expectations: the columns we do arithmetic on must be numeric, not text.
numeric_needed = ["annual_average_employees", "total_hours_worked", "total_deaths",
                  "total_dafw_cases", "total_djtr_cases", "total_other_cases",
                  "establishment_type", "size", "no_injuries_illnesses", "year_filing_for"]
types = dict(zip(described["column_name"], described["column_type"]))
print("\nInferred type of each column used in arithmetic or filtering:")
for col in numeric_needed + ["naics_code", "created_timestamp", "change_reason", "state"]:
    print(f"  {col:<28} {types[col]}")

,column_name,column_type,null
0,id,BIGINT,YES
1,establishment_name,VARCHAR,YES
2,establishment_id,VARCHAR,YES
3,ein,VARCHAR,YES
4,company_name,VARCHAR,YES
5,street_address,VARCHAR,YES
6,city,VARCHAR,YES
7,state,VARCHAR,YES
8,zip_code,VARCHAR,YES
9,naics_code,BIGINT,YES



PRD section 4 columns : 32
raw_300a columns      : 32
identical, same order : True
in PRD but not loaded : none
loaded but not in PRD : none

Inferred type of each column used in arithmetic or filtering:
  annual_average_employees     BIGINT
  total_hours_worked           DOUBLE
  total_deaths                 BIGINT
  total_dafw_cases             BIGINT
  total_djtr_cases             BIGINT
  total_other_cases            BIGINT
  establishment_type           BIGINT
  size                         BIGINT
  no_injuries_illnesses        BIGINT
  year_filing_for              BIGINT
  naics_code                   BIGINT
  created_timestamp            VARCHAR
  change_reason                VARCHAR
  state                        VARCHAR


In [4]:
# Three columns that the data dictionary describes as counts came back as VARCHAR:
# total_skin_disorders, total_hearing_loss, total_other_illnesses. Because sample_size = -1
# scanned every row, that is not a sampling artefact — there are genuinely non-numeric
# values in those columns. Find out what they are and how many. Nothing is altered here.

for col in ["total_skin_disorders", "total_hearing_loss", "total_other_illnesses"]:
    bad = con.sql(f"""
        SELECT {col} AS raw_value, COUNT(*) AS n
        FROM raw_300a
        WHERE {col} IS NOT NULL
          AND TRY_CAST({col} AS BIGINT) IS NULL
        GROUP BY 1 ORDER BY n DESC LIMIT 10
    """).df()
    total_bad = con.sql(f"""
        SELECT COUNT(*) FROM raw_300a
        WHERE {col} IS NOT NULL AND TRY_CAST({col} AS BIGINT) IS NULL
    """).fetchone()[0]
    print(f"{col}: {total_bad:,} non-numeric, non-null values")
    if total_bad:
        display(bad)

# None of these three columns feeds the TRIR numerator. The four recordable case-count
# columns are total_dafw_cases, total_djtr_cases, total_other_cases (and total_deaths),
# and all four loaded as BIGINT. Confirm that directly.
print("\nTypes of the columns that actually feed the metric:")
display(con.sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'raw_300a'
      AND column_name IN ('total_deaths','total_dafw_cases','total_djtr_cases',
                          'total_other_cases','total_hours_worked','annual_average_employees')
    ORDER BY column_name
""").df())

total_skin_disorders: 3 non-numeric, non-null values


,raw_value,n
0,02MAR26:16:16:00,2
1,03FEB26:17:33:00,1


total_hearing_loss: 3 non-numeric, non-null values


,raw_value,n
0,Manufacturing,2
1,Administrative and Support and Waste Managemen...,1


total_other_illnesses: 0 non-numeric, non-null values

Types of the columns that actually feed the metric:


,column_name,data_type
0,annual_average_employees,BIGINT
1,total_dafw_cases,BIGINT
2,total_deaths,BIGINT
3,total_djtr_cases,BIGINT
4,total_hours_worked,DOUBLE
5,total_other_cases,BIGINT


In [5]:
# The non-numeric values look like a column shift: a timestamp and a NAICS sector name
# landed in count columns. Pull those rows whole and look at them. Also resolve why
# total_other_illnesses is VARCHAR when every non-null value casts cleanly to an integer.

shifted = con.sql("""
    SELECT id, establishment_name, state, naics_code, industry_description,
           size, annual_average_employees, total_hours_worked,
           total_dafw_cases, total_djtr_cases, total_other_cases,
           total_skin_disorders, total_respiratory_conditions, total_poisonings,
           total_hearing_loss, total_other_illnesses, created_timestamp,
           change_reason, year_filing_for
    FROM raw_300a
    WHERE TRY_CAST(total_skin_disorders AS BIGINT) IS NULL AND total_skin_disorders IS NOT NULL
       OR TRY_CAST(total_hearing_loss   AS BIGINT) IS NULL AND total_hearing_loss   IS NOT NULL
    ORDER BY id
""").df()
print(f"rows with a non-numeric value in a count column: {len(shifted)}")
display(shifted.T)   # transposed: one column per bad row, easier to read

# Why is total_other_illnesses VARCHAR? Look at the distinct values that are not plain digits.
print("\ntotal_other_illnesses — distinct values that are not plain digits:")
display(con.sql("""
    SELECT total_other_illnesses AS raw_value,
           length(total_other_illnesses) AS len,
           COUNT(*) AS n
    FROM raw_300a
    WHERE total_other_illnesses IS NOT NULL
      AND NOT regexp_matches(total_other_illnesses, '^[0-9]+$')
    GROUP BY 1, 2 ORDER BY n DESC LIMIT 10
""").df())

rows with a non-numeric value in a count column: 3


,0,1,2
id,<NA>,<NA>,<NA>
establishment_name,Grand Prairie,Grand Prairie,Norwalk
state,1,1,1
naics_code,85,20,30
industry_description,2,1,2
size,0,0,0
annual_average_employees,0,1,0
total_hours_worked,0.0,0.0,0.0
total_dafw_cases,0,1,0
total_djtr_cases,0,0,0



total_other_illnesses — distinct values that are not plain digits:


,raw_value,len,n


In [6]:
# Resolution of the shifted rows.
#
# Checked outside DuckDB with Python's csv module: all 383,283 data rows have exactly 32
# fields, so the file is structurally valid CSV — that is why the load raised no error.
# The corruption is semantic and confined to the last 6 data lines of the file (383,279 -
# 383,284). Three records were split across six lines: three "right halves" (empty id, real
# values shifted left into the wrong columns, a sector name and an implausible date such as
# "Saturday, November 18, 2795" in the tail columns) and three "left halves" (identifying
# columns filled, everything from `state` onward empty).
#
# `total_other_illnesses` is VARCHAR for the same reason: two of those rows carry ZIP codes
# there, and "06854" has a leading zero, which DuckDB preserves as text rather than losing.
#
# Nothing is repaired here. This cell only counts the damage.

print("Left halves — id present, everything from `state` onward NULL:")
display(con.sql("""
    SELECT id, establishment_name, establishment_id, ein, company_name, street_address
    FROM raw_300a
    WHERE state IS NULL AND naics_code IS NULL AND year_filing_for IS NULL
      AND id IS NOT NULL
    ORDER BY id
""").df())

print("Right halves — id NULL:")
display(con.sql("""
    SELECT establishment_name AS shifted_city, state AS shifted_state,
           naics_code AS shifted_field, total_skin_disorders, total_hearing_loss,
           total_other_illnesses, created_timestamp
    FROM raw_300a WHERE id IS NULL
""").df())

for label, where in [("id IS NULL", "id IS NULL"),
                     ("year_filing_for IS NULL", "year_filing_for IS NULL"),
                     ("state IS NULL", "state IS NULL"),
                     ("either half of the split records",
                      "id IS NULL OR (state IS NULL AND naics_code IS NULL AND year_filing_for IS NULL)")]:
    n = con.sql(f"SELECT COUNT(*) FROM raw_300a WHERE {where}").fetchone()[0]
    print(f"rows where {label:<34}: {n}")

Left halves — id present, everything from `state` onward NULL:


,id,establishment_name,establishment_id,ein,company_name,street_address
0,3014415,City Carting Inc,727816,061200482,None,18 Meadow Street
1,3228570,Grand Prairie,1530137,None,Glass and Glazing Systems,1101 Fountain Pkwy
2,3228571,Dallas SC,1530138,None,Glass and Glazing Systems,1931 N. Great SW Parkway


Right halves — id NULL:


,shifted_city,shifted_state,shifted_field,total_skin_disorders,total_hearing_loss,total_other_illnesses,created_timestamp
0,Grand Prairie,1,85,02MAR26:16:16:00,Manufacturing,75050,"Saturday, November 18, 2795"
1,Grand Prairie,1,20,02MAR26:16:16:00,Manufacturing,75050,"Tuesday, November 10, 2809"
2,Norwalk,1,30,03FEB26:17:33:00,Administrative and Support and Waste Managemen...,06854,"Wednesday, January 2, 3439"


rows where id IS NULL                        : 3
rows where year_filing_for IS NULL           : 6
rows where state IS NULL                     : 3
rows where either half of the split records  : 6


In [7]:
# Shape of the raw table, and value counts for the three coded fields.
#
# `establishment_id` should identify a workplace. If distinct ids < row count, some
# establishments filed more than once — audit check 2 quantifies that below.
# The `size` counts test a specific claim in PRD section 4: legacy code 2 (20-249, split in
# 2023) should not appear in 2025 data.

n_raw = con.sql("SELECT COUNT(*) FROM raw_300a").fetchone()[0]
n_estab = con.sql("SELECT COUNT(DISTINCT establishment_id) FROM raw_300a").fetchone()[0]
n_estab_null = con.sql("SELECT COUNT(*) FROM raw_300a WHERE establishment_id IS NULL").fetchone()[0]
print(f"rows                              : {n_raw:,}")
print(f"distinct establishment_id         : {n_estab:,}")
print(f"rows minus distinct ids           : {n_raw - n_estab:,}")
print(f"rows with NULL establishment_id   : {n_estab_null:,}")
print(f"raw CSV on disk                   : {CSV_PATH.stat().st_size / 1048576:.2f} MB")
print(f"DuckDB file on disk               : {DB_PATH.stat().st_size / 1048576:.2f} MB")

for col, legend in [
    ("year_filing_for", {}),
    ("establishment_type", {1: "private", 2: "state government", 3: "local government"}),
    ("size", {1: "under 20 employees", 2: "LEGACY 20-249 (retired 2023)",
              21: "20-99", 22: "100-249", 3: "250+"}),
]:
    df = con.sql(f"""
        SELECT {col} AS value, COUNT(*) AS n,
               ROUND(100.0 * COUNT(*) / {n_raw}, 3) AS pct_of_raw
        FROM raw_300a GROUP BY 1 ORDER BY n DESC
    """).df()
    df["meaning"] = df["value"].map(lambda v: legend.get(v, ""))
    print(f"\n{col} value counts:")
    display(df)

rows                              : 383,283
distinct establishment_id         : 383,282
rows minus distinct ids           : 1
rows with NULL establishment_id   : 0
raw CSV on disk                   : 80.68 MB
DuckDB file on disk               : 163.76 MB

year_filing_for value counts:


,value,n,pct_of_raw,meaning
0,2025,383277,99.998,
1,<NA>,6,0.002,



establishment_type value counts:


,value,n,pct_of_raw,meaning
0,1,355344,92.711,private
1,2,16183,4.222,state government
2,3,11313,2.952,local government
3,<NA>,440,0.115,
4,0,3,0.001,



size value counts:


,value,n,pct_of_raw,meaning
0,21,151189,39.446,20-99
1,1,99658,26.001,under 20 employees
2,22,58704,15.316,100-249
3,3,40603,10.593,250+
4,2,33123,8.642,LEGACY 20-249 (retired 2023)
5,<NA>,3,0.001,
6,0,3,0.001,


## Section 2 — Look at the data

Null rate per column, a sample of rows, and the range of every column that feeds the metric.
Still nothing dropped and nothing altered.

In [8]:
# Null rate for every column, one row per column, highest first.
#
# DuckDB cannot count nulls across columns of mixed type in a single static query, so the
# SELECT is generated from the column list and run as one query returning one row per
# column. Empty strings are counted separately: an empty string is not NULL, but for a text
# column it is just as missing, and the distinction matters when deciding exclusions.

cols = con.sql("SELECT column_name, data_type FROM information_schema.columns "
               "WHERE table_name = 'raw_300a' ORDER BY ordinal_position").df()

parts = []
for name, dtype in zip(cols["column_name"], cols["data_type"]):
    blank = (f"COUNT(*) FILTER (WHERE trim({name}) = '')" if dtype == "VARCHAR" else "0")
    parts.append(f"""
        SELECT '{name}' AS column_name,
               '{dtype}' AS data_type,
               COUNT(*) FILTER (WHERE {name} IS NULL) AS n_null,
               {blank} AS n_blank
        FROM raw_300a""")

null_rates = con.sql(f"""
    WITH per_column AS ({' UNION ALL '.join(parts)})
    SELECT column_name, data_type, n_null,
           ROUND(100.0 * n_null / {n_raw}, 3) AS pct_null,
           n_blank,
           ROUND(100.0 * (n_null + n_blank) / {n_raw}, 3) AS pct_null_or_blank
    FROM per_column
    ORDER BY pct_null_or_blank DESC, column_name
""").df()
display(null_rates)

,column_name,data_type,n_null,pct_null,n_blank,pct_null_or_blank
0,change_reason,VARCHAR,369679,96.451,0,96.451
1,ein,VARCHAR,43317,11.302,0,11.302
2,industry_description,VARCHAR,26131,6.818,0,6.818
3,company_name,VARCHAR,19053,4.971,0,4.971
4,establishment_type,BIGINT,440,0.115,0,0.115
5,total_respiratory_conditions,BIGINT,6,0.002,0,0.002
6,year_filing_for,BIGINT,6,0.002,0,0.002
7,annual_average_employees,BIGINT,3,0.001,0,0.001
8,city,VARCHAR,3,0.001,0,0.001
9,created_timestamp,VARCHAR,3,0.001,0,0.001


In [9]:
# Ten sample rows, key columns only, so the shape of a record is visible.
#
# `establishment_name` appears here and only here. Per PRD section 4, OSHA does not validate
# these submissions, so no ranking or chart later in this project names an individual
# establishment or company.

display(con.sql("""
    SELECT establishment_name, state, naics_code, establishment_type, size,
           annual_average_employees, total_hours_worked, no_injuries_illnesses,
           total_deaths, total_dafw_cases, total_djtr_cases, total_other_cases,
           year_filing_for
    FROM raw_300a
    USING SAMPLE 10 ROWS (reservoir, 42)
""").df())

,establishment_name,state,naics_code,establishment_type,size,annual_average_employees,total_hours_worked,no_injuries_illnesses,total_deaths,total_dafw_cases,total_djtr_cases,total_other_cases,year_filing_for
0,New Jersey South 1800 Got Junk,NJ,562111,1,21,37,54000.0,1,0,1,0,7,2025
1,"Signature Flooring, Inc.",CA,238330,1,21,74,102975.0,2,0,0,0,0,2025
2,"FAREWAY STORES, INC. 983 GRIMES",IA,445110,1,22,98,70912.0,1,0,1,0,0,2025
3,34440 - BVLS - Naples,FL,561730,1,2,83,295694.0,1,0,0,0,1,2025
4,2140 - Tucson North,AZ,452112,1,3,271,271384.0,1,0,2,1,0,2025
5,Saegertown Family Practice,PA,621111,1,1,6,9755.0,2,0,0,0,0,2025
6,"Midwest Healthcare Linen Service, LLC",IA,812331,1,22,107,231118.0,1,0,0,0,3,2025
7,"Rescue Ambulance Services, Inc",PR,621910,1,1,16,19738.0,2,0,0,0,0,2025
8,63RD STREET,IL,488210,1,2,23,65017.0,1,0,2,0,0,2025
9,"Goldin & Stafford, LLC",DC,238910,1,21,83,185110.0,2,0,0,0,0,2025


In [10]:
# Range of every column that feeds the metric: min, median, max, plus counts of zeros and
# negatives. Median rather than mean, because a handful of very large establishments would
# drag a mean and hide the typical case.
#
# What to watch for: a minimum of 0 in total_hours_worked (TRIR would be undefined), any
# negative case count (impossible), and an implausibly large maximum.
#
# One SELECT per column, stacked with UNION ALL, so the aggregates share a single scan.

metric_cols = ["annual_average_employees", "total_hours_worked", "total_deaths",
               "total_dafw_cases", "total_djtr_cases", "total_other_cases"]

blocks = [f"""
    SELECT '{c}' AS column_name,
           MIN({c}) AS min_value,
           MEDIAN({c}) AS median_value,
           MAX({c}) AS max_value,
           COUNT(*) FILTER (WHERE {c} = 0) AS n_zero,
           COUNT(*) FILTER (WHERE {c} < 0) AS n_negative,
           COUNT(*) FILTER (WHERE {c} IS NULL) AS n_null
    FROM raw_300a""" for c in metric_cols]

ranges = con.sql(" UNION ALL ".join(blocks)).df()
# Preserve the listed order rather than whatever order the union returns.
ranges = ranges.set_index("column_name").loc[metric_cols].reset_index()
display(ranges)

,column_name,min_value,median_value,max_value,n_zero,n_negative,n_null
0,annual_average_employees,0.0,39.0,1.206674e+08,4468,0,3
1,total_hours_worked,0.0,66102.5,1.401420e+11,1837,0,3
2,total_deaths,0.0,0.0,1.600000e+01,382589,0,3
3,total_dafw_cases,0.0,0.0,1.025000e+03,250246,0,3
4,total_djtr_cases,0.0,0.0,6.214000e+03,279412,0,3
5,total_other_cases,0.0,0.0,6.504000e+03,257568,0,3


In [11]:
# The maxima above are not plausible, so look at the rows behind them before writing any
# audit rule. A single establishment cannot employ 120 million people or work 140 billion
# hours. `hours_per_employee` is the diagnostic: a full-time year is roughly 2,000 hours,
# so values in the hundreds of thousands mean the figure was entered in the wrong unit or
# is simply wrong.

print("Top 5 rows by annual_average_employees:")
display(con.sql("""
    SELECT annual_average_employees, total_hours_worked,
           ROUND(total_hours_worked / NULLIF(annual_average_employees, 0), 1) AS hours_per_employee,
           size, state, naics_code,
           total_dafw_cases + total_djtr_cases + total_other_cases AS recordable_cases
    FROM raw_300a ORDER BY annual_average_employees DESC NULLS LAST LIMIT 5
""").df())

print("\nTop 5 rows by total_hours_worked:")
display(con.sql("""
    SELECT total_hours_worked, annual_average_employees,
           ROUND(total_hours_worked / NULLIF(annual_average_employees, 0), 1) AS hours_per_employee,
           size, state, naics_code,
           total_dafw_cases + total_djtr_cases + total_other_cases AS recordable_cases
    FROM raw_300a ORDER BY total_hours_worked DESC NULLS LAST LIMIT 5
""").df())

# How far into the tail does implausibility reach? Percentiles of total_hours_worked.
print("\nPercentiles of total_hours_worked:")
display(con.sql("""
    SELECT quantile_cont(total_hours_worked, 0.50) AS p50,
           quantile_cont(total_hours_worked, 0.90) AS p90,
           quantile_cont(total_hours_worked, 0.99) AS p99,
           quantile_cont(total_hours_worked, 0.999) AS p999,
           MAX(total_hours_worked) AS max
    FROM raw_300a
""").df())

Top 5 rows by annual_average_employees:


,annual_average_employees,total_hours_worked,hours_per_employee,size,state,naics_code,recordable_cases
0,120667434,727.0,0.0,3,FL,237210,10
1,81170689,170689.0,0.0,21,MI,424480,1
2,32841346,15789.0,0.0,3,TX,541330,16
3,8527485,78.0,0.0,21,IL,485410,0
4,4883821,83821.0,0.0,21,MO,623220,0



Top 5 rows by total_hours_worked:


,total_hours_worked,annual_average_employees,hours_per_employee,size,state,naics_code,recordable_cases
0,1.401420e+11,137,1.022934e+09,2,NJ,325412,1
1,1.197590e+11,60,1.995983e+09,2,NJ,325412,0
2,1.005510e+11,147,6.840204e+08,22,NJ,493110,5
3,4.157766e+10,44,9.449468e+08,2,NJ,561910,0
4,4.003621e+10,22,1.819828e+09,2,NJ,325412,0



Percentiles of total_hours_worked:


,p50,p90,p99,p999,max
0,66102.5,405823.8,2841920.87,2.170335e+07,1.401420e+11


## Section 3 — Audit: 12 data-quality checks

`sql/02_audit.sql` runs the 12 checks from PRD section 6 as one query, one row per check.
A row can fail several checks, and each check is counted independently against all 383,283
raw rows, so these counts are not meant to sum to a total.

**No row is dropped and no value is altered.** Which of these become exclusions is a
decision for the M1 checkpoint, written into PRD section 7 before M2 starts.

In [12]:
# Run the 12 audit checks. The sector lookup path is passed through the run() helper as a
# {SECTORS_PATH} placeholder, so the .sql file carries no machine-specific absolute path.

audit = run("02_audit.sql", SECTORS_PATH=SQL_DIR / "naics_sectors.csv")
display(audit)

print(f"\nraw rows: {n_raw:,}")
print("Reminder: a row can fail several checks; these do not sum to a total.")

,check_no,description,rows_failing,pct_of_raw
0,1,year_filing_for is not 2025 (or is NULL),6,0.002
1,2,duplicate establishment_id (rows minus distinc...,1,0.000
2,3,total_hours_worked is NULL,3,0.001
3,4,total_hours_worked < 1000,9049,2.361
4,5,annual_average_employees is NULL or 0,4471,1.167
5,6,"naics_code NULL, not six digits, or sector not...",21,0.005
6,7,a case-count column is NULL or negative,3,0.001
7,8,recordable_cases > annual_average_employees,412,0.107
8,9,no_injuries_illnesses inconsistent with the ca...,135,0.035
9,10,"hours per employee above 5,000 or below 100",5243,1.368



raw rows: 383,283
Reminder: a row can fail several checks; these do not sum to a total.


In [13]:
# Check 2 drill-down: what does a duplicate actually look like?
#
# PRD section 6 asks for the change_reason distribution and whether duplicate rows differ in
# created_timestamp — that is what would let M2 keep the latest filing and drop the earlier
# one. Only 1 excess row exists, so this is a very small problem, but the mechanism still
# has to be understood before M2 writes a de-duplication rule.

print("change_reason distribution across all raw rows:")
display(con.sql(f"""
    SELECT COALESCE(change_reason, '(NULL)') AS change_reason,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / {n_raw}, 3) AS pct_of_raw
    FROM raw_300a GROUP BY 1 ORDER BY n DESC
""").df())

print("\nEstablishment ids appearing more than once:")
dup_ids = con.sql("""
    SELECT establishment_id, COUNT(*) AS n_rows
    FROM raw_300a GROUP BY 1 HAVING COUNT(*) > 1 ORDER BY n_rows DESC
""").df()
display(dup_ids)

print("\nEvery row for those establishment ids, side by side:")
display(con.sql("""
    SELECT id, establishment_id, establishment_name, state, naics_code, size,
           annual_average_employees, total_hours_worked,
           total_dafw_cases, total_djtr_cases, total_other_cases,
           created_timestamp, change_reason, year_filing_for
    FROM raw_300a
    WHERE establishment_id IN (
        SELECT establishment_id FROM raw_300a GROUP BY 1 HAVING COUNT(*) > 1
    )
    ORDER BY establishment_id, created_timestamp
""").df())

change_reason distribution across all raw rows:


,change_reason,n,pct_of_raw
0,(NULL),369679,96.451
1,0,3441,0.898
2,Initial Submission,1801,0.470
3,Uploaded Correct Locations and Data,121,0.032
4,yes,119,0.031
...,...,...,...
6394,wrong data entered,1,0.000
6395,Revised due to updated employee count. Prior e...,1,0.000
6396,incorrect field,1,0.000
6397,Had not removed a 2026 injury from reporting i...,1,0.000



Establishment ids appearing more than once:


,establishment_id,n_rows
0,TX,2



Every row for those establishment ids, side by side:


,id,establishment_id,establishment_name,state,naics_code,size,annual_average_employees,total_hours_worked,total_dafw_cases,total_djtr_cases,total_other_cases,created_timestamp,change_reason,year_filing_for
0,<NA>,TX,Grand Prairie,1,85,0,0,0.0,0,0,0,"Saturday, November 18, 2795",None,<NA>
1,<NA>,TX,Grand Prairie,1,20,0,1,0.0,1,0,0,"Tuesday, November 10, 2809",None,<NA>


In [14]:
# Check 10 drill-down: hours per employee, in bands.
#
# A full-time year is about 2,000 hours (40 hours x 50 weeks), so a plausible establishment
# sits in the 1,000-3,000 band. Below 100 means the hours figure is far too small for the
# headcount; above 5,000 means it is far too large. Both are the usual signature of a figure
# entered in the wrong unit, or of the two fields being mixed up.
#
# Rows where employees is NULL or 0 cannot produce a ratio at all; they are shown as their
# own band rather than quietly dropped, so every one of the 383,283 rows is accounted for.

display(con.sql(f"""
    WITH b AS (
        SELECT CASE
                 WHEN annual_average_employees IS NULL OR annual_average_employees = 0
                      OR total_hours_worked IS NULL THEN '(not computable)'
                 WHEN total_hours_worked / annual_average_employees < 100     THEN '1. < 100'
                 WHEN total_hours_worked / annual_average_employees < 1000    THEN '2. 100 - 1,000'
                 WHEN total_hours_worked / annual_average_employees < 3000    THEN '3. 1,000 - 3,000'
                 WHEN total_hours_worked / annual_average_employees <= 5000   THEN '4. 3,000 - 5,000'
                 ELSE '5. > 5,000'
               END AS band
        FROM raw_300a
    )
    SELECT band, COUNT(*) AS n_rows,
           ROUND(100.0 * COUNT(*) / {n_raw}, 3) AS pct_of_raw
    FROM b GROUP BY 1 ORDER BY band
""").df())

print("Bands 1 and 5 together are the rows counted by audit check 10.")

,band,n_rows,pct_of_raw
0,(not computable),4471,1.167
1,1. < 100,2619,0.683
2,"2. 100 - 1,000",34648,9.040
3,"3. 1,000 - 3,000",334723,87.331
4,"4. 3,000 - 5,000",4198,1.095
5,"5. > 5,000",2624,0.685


Bands 1 and 5 together are the rows counted by audit check 10.


In [15]:
# Check 11 drill-down: 34,806 rows (9.08%) is the largest failure count, so find out what is
# driving it. Compare each row's `size` code against the band implied by
# annual_average_employees, and count agreements and disagreements per size code.
#
# This matters for M2: PRD section 5 plans to use OSHA's `size` code as the size band. If the
# code disagrees with the headcount this often, that choice needs defending.

display(con.sql("""
    WITH b AS (
        SELECT size,
               CASE WHEN annual_average_employees < 20  THEN 1
                    WHEN annual_average_employees < 100 THEN 21
                    WHEN annual_average_employees < 250 THEN 22
                    ELSE 3 END AS implied_size
        FROM raw_300a
        WHERE annual_average_employees IS NOT NULL
    )
    SELECT COALESCE(CAST(size AS VARCHAR), '(NULL)') AS size_code,
           COUNT(*) AS n_rows,
           COUNT(*) FILTER (WHERE size = implied_size
                               OR (size = 2 AND implied_size IN (21, 22))) AS n_agree,
           COUNT(*) FILTER (WHERE NOT (size = implied_size
                               OR (size = 2 AND implied_size IN (21, 22)))
                               OR size IS NULL) AS n_disagree
    FROM b GROUP BY 1 ORDER BY n_disagree DESC
""").df())

print("\nWhere the disagreements land — reported size code vs the band implied by headcount:")
display(con.sql("""
    SELECT COALESCE(CAST(size AS VARCHAR), '(NULL)') AS reported_size,
           CASE WHEN annual_average_employees < 20  THEN '1  (<20)'
                WHEN annual_average_employees < 100 THEN '21 (20-99)'
                WHEN annual_average_employees < 250 THEN '22 (100-249)'
                ELSE '3  (250+)' END AS implied_band,
           COUNT(*) AS n_rows
    FROM raw_300a
    WHERE annual_average_employees IS NOT NULL
    GROUP BY 1, 2 ORDER BY n_rows DESC LIMIT 12
""").df())

,size_code,n_rows,n_agree,n_disagree
0,21,151189,140016,11173
1,3,40603,32609,7994
2,22,58704,50881,7823
3,1,99658,94921,4737
4,2,33123,30047,3076
5,0,3,0,3



Where the disagreements land — reported size code vs the band implied by headcount:


,reported_size,implied_band,n_rows
0,21,21 (20-99),140016
1,1,1 (<20),94921
2,22,22 (100-249),50881
3,3,3 (250+),32609
4,2,21 (20-99),23401
5,21,1 (<20),8512
6,2,22 (100-249),6646
7,22,21 (20-99),5922
8,1,21 (20-99),4106
9,3,22 (100-249),3291


In [16]:
# Check 6 drill-down: only 21 rows fail the NAICS test, but a number should not be quoted
# without knowing what is behind it. Also look at the undocumented establishment_type
# values, since PRD section 4 lists only 1, 2 and 3.

print("NAICS failures, by reason:")
display(con.sql("""
    SELECT CASE WHEN naics_code IS NULL THEN 'NULL naics_code'
                WHEN naics_code < 100000 OR naics_code > 999999 THEN 'not six digits'
                ELSE 'sector not in lookup' END AS reason,
           COUNT(*) AS n_rows,
           MIN(naics_code) AS min_code, MAX(naics_code) AS max_code
    FROM raw_300a
    WHERE naics_code IS NULL
       OR naics_code < 100000 OR naics_code > 999999
       OR naics_code // 10000 NOT IN (
            SELECT sector_code FROM read_csv_auto('{SECTORS}', header = true))
    GROUP BY 1 ORDER BY n_rows DESC
""".replace("{SECTORS}", str(SQL_DIR / "naics_sectors.csv"))).df())

print("\nThe distinct offending naics_code values:")
display(con.sql("""
    SELECT naics_code, COUNT(*) AS n_rows
    FROM raw_300a
    WHERE naics_code IS NOT NULL
      AND (naics_code < 100000 OR naics_code > 999999
           OR naics_code // 10000 NOT IN (
                SELECT sector_code FROM read_csv_auto('{SECTORS}', header = true)))
    GROUP BY 1 ORDER BY n_rows DESC LIMIT 20
""".replace("{SECTORS}", str(SQL_DIR / "naics_sectors.csv"))).df())

print("\nnaics_year value counts (which NAICS vintage each row was coded against):")
display(con.sql(f"""
    SELECT naics_year, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / {n_raw}, 2) AS pct_of_raw
    FROM raw_300a GROUP BY 1 ORDER BY n DESC
""").df())

NAICS failures, by reason:


,reason,n_rows,min_code,max_code
0,sector not in lookup,15,178102,999999
1,not six digits,3,20,85
2,NULL naics_code,3,<NA>,<NA>



The distinct offending naics_code values:


,naics_code,n_rows
0,283220,5
1,999999,4
2,951299,1
3,178102,1
4,508425,1
5,20,1
6,85,1
7,361392,1
8,30,1
9,598113,1



naics_year value counts (which NAICS vintage each row was coded against):


,naics_year,n,pct_of_raw
0,2022,231474,60.39
1,2012,139490,36.39
2,2017,12143,3.17
3,0,167,0.04
4,<NA>,3,0.00
5,2025,3,0.00
6,69575,1,0.00
7,37678,1,0.00
8,180175,1,0.00


In [17]:
# Independent structural check of the raw file, outside DuckDB.
#
# The load raised no error, but "no error" is not the same as "nothing wrong". Parse the CSV
# with Python's own reader and count the fields on every line. If any line had a field count
# other than 32, DuckDB's loader and this reader could disagree about where rows begin and
# end, and rows could be silently mangled. This also locates the six corrupted lines by
# line number so the finding is reproducible from the notebook.

import csv

field_counts = {}
suspect_lines = []
with open(CSV_PATH, newline="", encoding="utf-8", errors="replace") as fh:
    reader = csv.reader(fh)
    header = next(reader)
    for lineno, row in enumerate(reader, start=2):
        field_counts[len(row)] = field_counts.get(len(row), 0) + 1
        # A row whose first field (id) is empty, or whose year column is empty.
        if row[0] == "" or row[31] == "":
            suspect_lines.append(lineno)

print(f"header fields                : {len(header)}")
print(f"field-count distribution     : {field_counts}")
print(f"data lines parsed            : {sum(field_counts.values()):,}")
print(f"rows loaded into raw_300a    : {n_raw:,}")
print(f"loader and csv module agree  : {sum(field_counts.values()) == n_raw}")
print(f"lines with empty id or year  : {suspect_lines}")
print(f"last data line in the file   : {1 + sum(field_counts.values()):,}")

header fields                : 32
field-count distribution     : {32: 383283}
data lines parsed            : 383,283
rows loaded into raw_300a    : 383,283
loader and csv module agree  : True
lines with empty id or year  : [383279, 383280, 383281, 383282, 383283, 383284]
last data line in the file   : 383,284


## Three things that look wrong

There are more than three. All of them are listed, worst first. Every number below comes
from a cell above in this notebook — nothing here is estimated, and nothing has been fixed.

### 1. The last six lines of the file are three records torn in half

Cell 20 parses the raw CSV independently of DuckDB: all 383,283 data lines have exactly 32
fields, and the loader and Python's `csv` module agree on the row count, so nothing was
silently mangled at load time. But the six lines with an empty `id` or an empty
`year_filing_for` are lines **383,279–383,284** — the *last six data lines in the file*.

Cell 7 shows what they are: three "left halves" (`id`, `establishment_name`,
`establishment_id`, `ein`, `company_name`, `street_address` filled, everything from `state`
onward empty) and three "right halves" (empty `id`, real values shifted left into the wrong
columns). "Grand Prairie" appears as an `establishment_name` on one line and as a `city` on
another. The right halves carry a `created_timestamp` of *"Saturday, November 18, 2795"*,
*"Tuesday, November 10, 2809"* and *"Wednesday, January 2, 3439"*.

Two consequences that would otherwise be invisible:

- **`total_skin_disorders`, `total_hearing_loss` and `total_other_illnesses` loaded as text,
  not integers** (cell 4). Cell 5 shows why: 3 values like `02MAR26:16:16:00` and 3 like
  `Manufacturing` sit in count columns, and a ZIP code `06854` — leading zero preserved —
  sits in `total_other_illnesses`. `sample_size = -1` scanned every row, so this is real,
  not a sampling artefact. None of the four columns feeding the metric is affected; all six
  metric columns are numeric (cell 5).
- **The single "duplicate" in audit check 2 is not a duplicate.** Cell 16 shows the only
  repeated `establishment_id` is the literal string `"TX"` — a state code shifted into the
  id column on two of these six rows. Across 383,283 rows there are **zero genuine duplicate
  establishments**, which is a stronger result than check 2's count of 1 suggests, and it
  means M2 needs no de-duplication rule at all.

### 2. The legacy `size` code 2 is in 33,123 rows (8.642%)

PRD section 4 states that code 2 (20–249 employees) was split in 2023 and "should not appear
in 2025 data; flag it if it does." It appears in roughly **1 row in 12** (cell 8, audit check
12). This is not a rounding-error problem: it breaks the four-band size analysis planned for
M2, because the legacy band **overlaps two current bands** (20–99 and 100–249) rather than
sitting beside them. A row coded 2 cannot be assigned to either without using
`annual_average_employees` instead — and cell 18 shows 23,401 of them imply 20–99 and 6,646
imply 100–249. This needs a decision before M2, not during it.

### 3. Employee and hours values that are physically impossible

Cell 12: `annual_average_employees` reaches **120,667,434** and `total_hours_worked` reaches
**140,142,000,000**. One establishment cannot employ 120 million people — that is over half
the US labour force — or work 140 billion hours. Cell 13 shows the maximum is about **6,000
times the 99.9th percentile** (21,703,350 hours).

Cell 13 also shows the likely mechanism, and it is specific: in at least two rows the
employee field appears to be the employee count *concatenated with the hours figure* —
`81170689` employees against `170689` hours ("81" followed by the hours), and `4883821`
against `83821` hours ("48" followed by the hours). The top hours rows are also suspiciously
homogeneous: four of the top five are New Jersey, three share NAICS 325412, and four carry
`size = 2`.

These rows would produce a TRIR near zero and, because M2 aggregates by summing cases and
summing hours, **a single one of them could dominate an entire sector's denominator.** This
is the finding with the largest effect on the result.

### 4. 9,049 rows (2.361%) report under 1,000 hours worked, and 1,837 report exactly zero

Audit check 4 and cell 12. Zero hours makes TRIR undefined (division by zero); a few hundred
hours makes it enormous and meaningless. Cell 17 puts this in context: 87.331% of rows sit in
the plausible 1,000–3,000 hours-per-employee band, so the problem is a real tail, not the
norm.

### 5. 4,471 rows (1.167%) have no usable employee count

Audit check 5: `annual_average_employees` is NULL (3) or zero (4,468, cell 12). These rows
cannot be size-banded or sanity-checked against their case counts.

### 6. 5,243 rows (1.368%) have implausible hours per employee

Audit check 10, decomposed in cell 17: 2,619 rows below 100 hours per employee and 2,624
above 5,000. The bands reconcile exactly to check 10's count. A full-time year is about 2,000
hours, so both tails point at a units or data-entry error rather than an unusual workplace.

### 7. 412 rows (0.107%) report more recordable cases than employees

Audit check 8. Not strictly impossible — headcount is an annual average and one worker can be
injured more than once — but at this ratio it is far more likely to be an error.

### 8. 135 rows (0.035%) contradict their own "no injuries" flag

Audit check 9: `no_injuries_illnesses` says 2 ("had none") while cases are positive, or says 1
("had cases") while cases are zero. A self-inconsistent form is a signal about how carefully
the whole row was filled in.

### 9. The sector lookup is applied to three different NAICS vintages

Cell 19: PRD section 5 specifies a **2022 NAICS** lookup, but only 60.39% of rows are coded
against 2022. **139,490 rows (36.39%) use NAICS 2012** and 12,143 (3.17%) use 2017, plus 167
rows with `naics_year = 0`. Two-digit sector codes are largely stable across these vintages,
so the sector mapping mostly survives — but this belongs in the README's limits section, not
in a footnote.

### 10. Small structural oddities worth recording

- **Audit check 11 is largely legitimate, and should not be treated as corruption.** 34,806
  rows (9.081%) have a `size` code disagreeing with the band implied by
  `annual_average_employees`, but PRD section 5 already warns that `size` is *peak* headcount
  while `annual_average_employees` is the *average*, so a seasonal employer differs honestly.
  Cell 18 shows the disagreements spread across every size code. The one sub-case that still
  looks wrong is **2,701 rows coded `size = 3` (250+) that average under 20 employees** — a
  more than twelvefold gap.
- **`change_reason` is free text, not a coded field**: 6,399 distinct values (cell 16),
  including `0`, `yes`, `Initial Submission` and whole sentences. 96.451% are NULL. It cannot
  carry any automated logic in M2.
- **`establishment_type` has 440 NULLs (0.115%) and 3 zeros** (cell 8); PRD section 4
  documents only 1, 2 and 3. The 3 zeros are the corrupted rows; the 440 NULLs are not.
- **Only 21 rows fail the NAICS check** (check 6, cell 19), of which 4 carry the placeholder
  `999999` and 3 are the corrupted rows. This column is in far better shape than expected.

---

# Milestone 2 — Clean table and analysis

The exclusion list and the recordable-cases definition were locked by Ashvi at the M1
checkpoint and written into `PRD.md` sections 5 and 7 before any of the SQL below was
written. In short:

- **Excluded, 8 steps:** M1 checks 1, 2, 3, 4, 5, 6, 7, 8 and 10 (checks 3 and 4 merge into
  step 3; check 10 is promoted from a flag to step 8).
- **Flagged, never excluded:** M1 checks 9, 11, 12.
- **`recordable_cases = dafw + djtr + other`** — OSHA Total Case Rate, deaths excluded and
  carried as their own count.
- **Size bands derived from `annual_average_employees`**, not OSHA's `size` code.
- **Establishments reporting zero recordable cases stay in.** They contribute hours to every
  denominator; removing them would inflate every rate in this project.

## Section 4 — Staging and the clean table

In [18]:
# Before writing the staging query: what does created_timestamp actually look like, and what
# does TRY_CAST(... AS TIMESTAMP) do to it?
#
# The de-duplication step orders by this column to keep the latest submission. If the cast
# returns NULL, the ordering degrades to the `id DESC` tiebreaker. That is safe here only
# because M1 established there are no genuine duplicates — but the behaviour has to be seen,
# not assumed, because the rule stays in the pipeline for future years' files.

print("Most common created_timestamp formats:")
display(con.sql("""
    SELECT created_timestamp,
           TRY_CAST(created_timestamp AS TIMESTAMP) AS cast_result,
           COUNT(*) AS n
    FROM raw_300a
    GROUP BY 1, 2 ORDER BY n DESC LIMIT 8
""").df())

print("\nHow many values cast successfully?")
display(con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(*) FILTER (WHERE created_timestamp IS NULL) AS n_null,
           COUNT(*) FILTER (WHERE created_timestamp IS NOT NULL
                              AND TRY_CAST(created_timestamp AS TIMESTAMP) IS NULL)
               AS n_cast_fails,
           COUNT(*) FILTER (WHERE TRY_CAST(created_timestamp AS TIMESTAMP) IS NOT NULL)
               AS n_cast_ok,
           ROUND(100.0 * COUNT(*) FILTER (WHERE TRY_CAST(created_timestamp AS TIMESTAMP)
                 IS NOT NULL) / {n_raw}, 3) AS pct_cast_ok
    FROM raw_300a
""").df())

print("\nRange of the values that do cast:")
display(con.sql("""
    SELECT MIN(TRY_CAST(created_timestamp AS TIMESTAMP)) AS earliest,
           MAX(TRY_CAST(created_timestamp AS TIMESTAMP)) AS latest
    FROM raw_300a
""").df())

Most common created_timestamp formats:


,created_timestamp,cast_result,n
0,2/27/2026,NaT,23405
1,3/2/2026,NaT,20292
2,2/25/2026,NaT,19726
3,2/26/2026,NaT,19186
4,2/19/2026,NaT,17854
5,3/6/2026,NaT,17160
6,2/24/2026,NaT,14879
7,2/23/2026,NaT,10744



How many values cast successfully?


,n_rows,n_null,n_cast_fails,n_cast_ok,pct_cast_ok
0,383283,3,383280,0,0.0



Range of the values that do cast:


,earliest,latest
0,NaT,NaT


In [19]:
# Build the staged and clean tables, then the waterfall.
#
# 03_stage.sql tags every raw row with the FIRST exclusion step it fails, so each row lands
# on exactly one step. That is what makes the reconciliation below hold by construction
# rather than by luck: raw = clean + sum(rows dropped per step).

run("03_stage.sql", SECTORS_PATH=SQL_DIR / "naics_sectors.csv")
run("04_clean.sql")

n_staged = con.sql("SELECT COUNT(*) FROM staged_300a").fetchone()[0]
n_clean = con.sql("SELECT COUNT(*) FROM clean_300a").fetchone()[0]
print(f"staged rows : {n_staged:,}   (must equal raw: {n_staged == n_raw})")
print(f"clean rows  : {n_clean:,}")

waterfall = run("05_waterfall.sql")
display(waterfall)

# Hard assertion: this either passes or the pipeline is wrong.
assert waterfall.rows_dropped.sum() == n_raw, (
    f"waterfall does not reconcile: {waterfall.rows_dropped.sum():,} vs {n_raw:,} raw rows")
print("\nwaterfall reconciles: "
      f"{waterfall.rows_dropped.sum():,} == {n_raw:,} raw rows")

n_dropped = n_raw - n_clean
print(f"\nraw {n_raw:,}  ->  clean {n_clean:,}   "
      f"({n_dropped:,} rows excluded, {100.0 * n_dropped / n_raw:.2f}%)")

# Step 2 is expected to be 0; PRD section 7 says stop and report if it is not.
step2 = int(waterfall.loc[waterfall.step == 2, "rows_dropped"].iloc[0])
print(f"\nstep 2 (duplicate submissions): {step2}  "
      f"-> {'as expected' if step2 == 0 else 'UNEXPECTED — STOP AND REPORT'}")

OUTPUT_DIR.joinpath("tables").mkdir(parents=True, exist_ok=True)
waterfall.to_csv(OUTPUT_DIR / "tables" / "drop_waterfall.csv", index=False)
print(f"saved -> {OUTPUT_DIR / 'tables' / 'drop_waterfall.csv'}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

staged rows : 383,283   (must equal raw: True)
clean rows  : 369,996


,step,rule,rows_dropped
0,1,year_filing_for missing or not 2025,6
1,2,duplicate submission for the same establishmen...,0
2,3,"total_hours_worked NULL or below 1,000",9046
3,4,annual_average_employees NULL or 0,221
4,5,naics_code invalid or sector not in the lookup,15
5,6,a case-count column NULL or negative,0
6,7,recordable cases greater than annual average e...,146
7,8,"hours per employee below 100 or above 5,000",3853
8,99,clean rows kept,369996



waterfall reconciles: 383,283 == 383,283 raw rows

raw 383,283  ->  clean 369,996   (13,287 rows excluded, 3.47%)

step 2 (duplicate submissions): 0  -> as expected
saved -> /Users/ashvijain/Leading Indicator/output/tables/drop_waterfall.csv


In [20]:
# What was actually removed at each step. Two example rows per step, chosen as the most
# extreme by the quantity that step tests, so the examples show the failure rather than a
# borderline case.
#
# Note on the counts: each row is attributed to the FIRST step it fails, so these are smaller
# than the independent M1 audit counts. Step 4 drops 221 rows where M1 check 5 counted 4,471
# zero/NULL-employee rows, because most of those also had hours below 1,000 and were already
# taken by step 3. That is the intended behaviour — it is what makes the waterfall reconcile.

order_by = {
    1: "year_filing_for NULLS FIRST",
    2: "submission_rank DESC",
    3: "total_hours_worked NULLS FIRST",
    4: "annual_average_employees NULLS FIRST",
    5: "naics_txt NULLS FIRST",
    6: "total_dafw_cases NULLS FIRST",
    7: "recordable_cases - annual_average_employees DESC",
    8: "hours_per_employee DESC",
}

for step in range(1, 9):
    n = con.sql(f"SELECT COUNT(*) FROM staged_300a WHERE drop_step = {step}").fetchone()[0]
    rule = waterfall.loc[waterfall.step == step, "rule"].iloc[0]
    print(f"\n--- step {step}: {rule}  ({n:,} rows) ---")
    if n == 0:
        print("    (no rows — nothing to show)")
        continue
    display(con.sql(f"""
        SELECT establishment_id, state, naics_code, size,
               annual_average_employees AS employees, total_hours_worked AS hours,
               ROUND(hours_per_employee, 3) AS hrs_per_emp,
               total_deaths AS deaths, recordable_cases, year_filing_for
        FROM staged_300a WHERE drop_step = {step}
        ORDER BY {order_by[step]} LIMIT 2
    """).df())


--- step 1: year_filing_for missing or not 2025  (6 rows) ---


,establishment_id,state,naics_code,size,employees,hours,hrs_per_emp,deaths,recordable_cases,year_filing_for
0,727816,None,<NA>,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>
1,CT,1,30,0,0,0.0,NaN,0,0,<NA>



--- step 2: duplicate submission for the same establishment and year (kept the latest)  (0 rows) ---
    (no rows — nothing to show)

--- step 3: total_hours_worked NULL or below 1,000  (9,046 rows) ---


,establishment_id,state,naics_code,size,employees,hours,hrs_per_emp,deaths,recordable_cases,year_filing_for
0,1469305,CT,624229,1,0,0.0,NaN,0,0,2025
1,1469741,NE,562111,1,0,0.0,NaN,0,0,2025



--- step 4: annual_average_employees NULL or 0  (221 rows) ---


...

,establishment_id,state,naics_code,size,employees,hours,hrs_per_emp,deaths,recordable_cases,year_filing_for
0,1114603,CA,178102,21,84,219959.0,2618.56,0,0,2025
1,960924,CT,283220,1,1,1967.0,1967.00,0,0,2025



--- step 6: a case-count column NULL or negative  (0 rows) ---
    (no rows — nothing to show)

--- step 7: recordable cases greater than annual average employees  (146 rows) ---


,establishment_id,state,naics_code,size,employees,hours,hrs_per_emp,deaths,recordable_cases,year_filing_for
0,684469,OH,621111,3,541,816064.0,1508.436,0,2192,2025
1,1306305,TX,541940,3,26,3705340.0,142513.077,0,308,2025



--- step 8: hours per employee below 100 or above 5,000  (3,853 rows) ---


,establishment_id,state,naics_code,size,employees,hours,hrs_per_emp,deaths,recordable_cases,year_filing_for
0,949631,NJ,325412,2,7,1.519751e+10,2.171073e+09,0,0,2025
1,949586,NJ,325412,2,60,1.197590e+11,1.995983e+09,0,0,2025


In [21]:
# Direct check that step 8 removed the employee/hours concatenation rows in BOTH directions.
# These are the specific rows identified in M1 (cell 13): the employee field appears to be the
# real headcount with the hours figure appended, e.g. 81,170,689 employees against 170,689
# hours ("81" + "170689"), and 4,883,821 against 83,821 hours ("48" + "83821").

print("The M1 extreme rows, and which step removed each:")
display(con.sql("""
    SELECT annual_average_employees AS employees, total_hours_worked AS hours,
           ROUND(hours_per_employee, 4) AS hrs_per_emp, drop_step, state, naics_code
    FROM staged_300a
    ORDER BY annual_average_employees DESC NULLS LAST LIMIT 5
""").df())

display(con.sql("""
    SELECT total_hours_worked AS hours, annual_average_employees AS employees,
           ROUND(hours_per_employee, 1) AS hrs_per_emp, drop_step, state, naics_code
    FROM staged_300a
    ORDER BY total_hours_worked DESC NULLS LAST LIMIT 5
""").df())

# Did anything implausible survive into the clean table?
print("\nRanges in the clean table — everything below must now be plausible:")
display(con.sql("""
    SELECT MIN(annual_average_employees) AS min_emp, MAX(annual_average_employees) AS max_emp,
           MIN(total_hours_worked) AS min_hours, MAX(total_hours_worked) AS max_hours,
           ROUND(MIN(hours_per_employee), 2) AS min_hrs_per_emp,
           ROUND(MAX(hours_per_employee), 2) AS max_hrs_per_emp,
           ROUND(MIN(trir), 4) AS min_trir, ROUND(MAX(trir), 2) AS max_trir
    FROM clean_300a
""").df())

n_extreme = con.sql("""
    SELECT COUNT(*) FROM staged_300a
    WHERE drop_step = 8 AND hours_per_employee < 100
""").fetchone()[0]
n_high = con.sql("""
    SELECT COUNT(*) FROM staged_300a
    WHERE drop_step = 8 AND hours_per_employee > 5000
""").fetchone()[0]
print(f"\nstep 8 breakdown: {n_extreme:,} rows below 100 hours/employee, "
      f"{n_high:,} rows above 5,000 (total {n_extreme + n_high:,})")

The M1 extreme rows, and which step removed each:


,employees,hours,hrs_per_emp,drop_step,state,naics_code
0,120667434,727.0,0.0000,3,FL,237210
1,81170689,170689.0,0.0021,8,MI,424480
2,32841346,15789.0,0.0005,8,TX,541330
3,8527485,78.0,0.0000,3,IL,485410
4,4883821,83821.0,0.0172,8,MO,623220


,hours,employees,hrs_per_emp,drop_step,state,naics_code
0,1.401420e+11,137,1.022934e+09,8,NJ,325412
1,1.197590e+11,60,1.995983e+09,8,NJ,325412
2,1.005510e+11,147,6.840204e+08,8,NJ,493110
3,4.157766e+10,44,9.449468e+08,8,NJ,561910
4,4.003621e+10,22,1.819828e+09,8,NJ,325412



Ranges in the clean table — everything below must now be plausible:


,min_emp,max_emp,min_hours,max_hours,min_hrs_per_emp,max_hrs_per_emp,min_trir,max_trir
0,1,138396,1000.0,455430263.0,100.0,5000.0,0.0,598.8



step 8 breakdown: 1,262 rows below 100 hours/employee, 2,591 rows above 5,000 (total 3,853)


## Section 5 — Aggregation

Every rate below is `SUM(recordable_cases) * 200000.0 / SUM(total_hours_worked)` — cases
summed, hours summed, divided once. Never an average of per-establishment rates: that would
weight a 10-person shop the same as a 5,000-person plant. Every aggregate row carries
`n_establishments`, `total_hours` and `total_cases` so the exposure behind each rate is
visible.

In [22]:
# Overall figures for the clean table.
#
# The zero-case count is the number that protects the whole project: those establishments are
# KEPT. They reported no recordable cases but still worked hours, and those hours belong in
# every denominator. Filtering them out would inflate every rate here.

overall = run("06_overall.sql")
display(overall.T)

o = overall.iloc[0]
print(f"clean establishments        : {int(o.n_establishments):,}")
print(f"employees (sum of averages) : {int(o.employees):,}")
print(f"total hours worked          : {o.total_hours:,.0f}")
print(f"recordable cases            : {int(o.total_cases):,}")
print(f"deaths (reported separately): {int(o.deaths):,}")
print(f"overall TRIR                : {o.trir:.3f} recordable cases per 100 full-time workers")
print(f"overall DART                : {o.dart:.3f}")
print(f"establishments with zero recordable cases: {int(o.n_zero_case_establishments):,} "
      f"({o.pct_zero_case:.2f}% of clean rows) — kept, not filtered")

OUT_T = OUTPUT_DIR / "tables"
overall.to_csv(OUT_T / "overall.csv", index=False)
print(f"\nsaved -> {OUT_T / 'overall.csv'}")

,0
n_establishments,3.699960e+05
employees,4.616517e+07
total_hours,7.781725e+10
total_cases,1.352105e+06
deaths,7.450000e+02
trir,3.475078e+00
dart,2.197790e+00
n_zero_case_establishments,1.602750e+05
pct_zero_case,4.332000e+01


clean establishments        : 369,996
employees (sum of averages) : 46,165,170
total hours worked          : 77,817,253,795
recordable cases            : 1,352,105
deaths (reported separately): 745
overall TRIR                : 3.475 recordable cases per 100 full-time workers
overall DART                : 2.198
establishments with zero recordable cases: 160,275 (43.32% of clean rows) — kept, not filtered

saved -> /Users/ashvijain/Leading Indicator/output/tables/overall.csv


In [23]:
# Sector rankings. One query produces both rankings, so they cannot drift apart.
# Saved twice: sector_by_rate.csv (ordered by rate) and sector_by_count.csv (ordered by count).

sector = run("07_sector_rankings.sql")
display(sector)

print(f"\nsectors present            : {len(sector)}")
print(f"sectors below n = 30       : {(sector.n_establishments < 30).sum()}")
print(f"establishments accounted for: {int(sector.n_establishments.sum()):,} "
      f"(clean table: {n_clean:,}) -> match: {int(sector.n_establishments.sum()) == n_clean}")

sector.sort_values("trir", ascending=False).to_csv(OUT_T / "sector_by_rate.csv", index=False)
sector.sort_values("total_cases", ascending=False).to_csv(OUT_T / "sector_by_count.csv", index=False)
print(f"saved -> {OUT_T / 'sector_by_rate.csv'}")
print(f"saved -> {OUT_T / 'sector_by_count.csv'}")

,sector,n_establishments,employees,total_hours,total_cases,deaths,trir,dart,rank_by_rate,rank_by_count,note
0,Public Administration,7877,1120102.0,1.919418e+09,53883.0,33.0,5.614515,3.162417,1,7,
1,Transportation and Warehousing,39658,5192320.0,8.800714e+09,239852.0,134.0,5.450739,4.511293,2,2,
2,"Arts, Entertainment, and Recreation",3687,786647.0,9.926831e+08,22768.0,9.0,4.587164,3.015665,3,10,
3,Retail Trade,59108,5927879.0,8.562892e+09,192744.0,39.0,4.501843,2.830913,4,4,
4,Accommodation and Food Services,17464,1369443.0,1.815814e+09,40393.0,23.0,4.449023,2.779249,5,8,
5,Health Care and Social Assistance,44947,10304120.0,1.554801e+10,333819.0,38.0,4.294043,2.065976,6,1,
6,"Agriculture, Forestry, Fishing and Hunting",4768,453285.0,7.628913e+08,15811.0,20.0,4.145020,2.739053,7,12,
7,Wholesale Trade,29831,1809555.0,3.459331e+09,60235.0,41.0,3.482465,2.701678,8,6,
8,Real Estate and Rental and Leasing,11733,391189.0,6.124633e+08,9973.0,12.0,3.256685,1.977261,9,15,
9,Educational Services,2904,1173814.0,1.554891e+09,20855.0,4.0,2.682503,1.101685,10,11,



sectors present            : 20
sectors below n = 30       : 0
establishments accounted for: 369,996 (clean table: 369,996) -> match: True
saved -> /Users/ashvijain/Leading Indicator/output/tables/sector_by_rate.csv
saved -> /Users/ashvijain/Leading Indicator/output/tables/sector_by_count.csv


In [24]:
# Size bands, derived from annual_average_employees (PRD section 5).
#
# pct_agree_with_osha_size is the validation statistic that replaced M1 audit check 11: the
# share of non-legacy rows whose OSHA `size` code matches the derived band. Legacy code 2 is
# excluded from that percentage because it spans two bands and could never match one; its
# count is reported separately.

size_band = run("08_size_band.sql")
display(size_band)

print(f"establishments accounted for: {int(size_band.n_establishments.sum()):,} "
      f"(clean: {n_clean:,}) -> match: {int(size_band.n_establishments.sum()) == n_clean}")
print(f"clean rows still carrying legacy size code 2: "
      f"{int(size_band.n_legacy_size_code.sum()):,} "
      f"({100.0 * size_band.n_legacy_size_code.sum() / n_clean:.2f}% of clean rows)")

size_band.to_csv(OUT_T / "size_band.csv", index=False)
print(f"saved -> {OUT_T / 'size_band.csv'}")

,size_band,size_order,n_establishments,employees,total_hours,total_cases,deaths,trir,dart,n_legacy_size_code,pct_agree_with_osha_size,note
0,Under 20,1,99492,877054.0,1.486378e+09,30964.0,38.0,4.166369,2.613332,2296,88.62,
1,20-99,2,173400,8149308.0,1.410659e+10,277306.0,250.0,3.931582,2.544528,23192,92.18,
2,100-249,3,62586,9563457.0,1.612864e+10,307143.0,208.0,3.808667,2.534337,6588,89.83,
3,250+,4,34518,27575351.0,4.609565e+10,736692.0,249.0,3.196362,1.960523,605,94.99,


establishments accounted for: 369,996 (clean: 369,996) -> match: True
clean rows still carrying legacy size code 2: 32,681 (8.83% of clean rows)
saved -> /Users/ashvijain/Leading Indicator/output/tables/size_band.csv


In [25]:
# States. All states saved; the top-10 list is built only from states with at least 30
# establishments, per PRD section 5. States below the threshold stay in state_all.csv marked
# "insufficient n" — shown, never silently dropped.

state = run("09_state.sql")
print(f"rows (states / territories): {len(state)}")
print(f"establishments accounted for: {int(state.n_establishments.sum()):,} "
      f"(clean: {n_clean:,}) -> match: {int(state.n_establishments.sum()) == n_clean}")

below = state[state.note == "insufficient n"]
print(f"\nstates below n = 30: {len(below)}")
if len(below):
    display(below[["state", "n_establishments", "total_hours", "total_cases", "trir", "note"]])

state.to_csv(OUT_T / "state_all.csv", index=False)

state_top10 = (state[state.n_establishments >= 30]
               .sort_values("trir", ascending=False)
               .head(10)
               .reset_index(drop=True))
print("\nTop 10 states by recordable-injury rate (n >= 30 establishments):")
display(state_top10)
state_top10.to_csv(OUT_T / "state_top10.csv", index=False)
print(f"saved -> {OUT_T / 'state_all.csv'}")
print(f"saved -> {OUT_T / 'state_top10.csv'}")

rows (states / territories): 61
establishments accounted for: 369,996 (clean: 369,996) -> match: True

states below n = 30: 7


,state,n_establishments,total_hours,total_cases,trir,note
0,AE,1,62304.0,4.0,12.840267,insufficient n
2,PW,2,8097408.0,226.0,5.582033,insufficient n
3,MH,1,402612.0,11.0,5.464318,insufficient n
50,AA,2,822598.0,12.0,2.917586,insufficient n
56,MP,13,1558559.0,17.0,2.181502,insufficient n
59,AS,5,377148.0,2.0,1.060592,insufficient n
60,FM,1,8455.0,0.0,0.000000,insufficient n



Top 10 states by recordable-injury rate (n >= 30 establishments):


,state,n_establishments,employees,total_hours,total_cases,deaths,trir,dart,rank_by_rate,note
0,ME,1843,155920.0,2.543609e+08,7939.0,2.0,6.242311,4.170452,2,
1,MT,1194,87188.0,1.471910e+08,3830.0,0.0,5.204122,3.100733,5,
2,VT,731,65186.0,1.057300e+08,2716.0,0.0,5.137616,3.087109,6,
3,WA,8550,1008605.0,1.687002e+09,41121.0,6.0,4.875038,3.389207,7,
4,ID,2318,204872.0,3.525600e+08,8080.0,1.0,4.583617,2.786476,8,
5,OR,6157,603775.0,1.001660e+09,22727.0,9.0,4.537868,3.044546,9,
6,CT,4797,595672.0,9.458310e+08,21202.0,5.0,4.483254,2.958245,10,
7,SD,1355,118127.0,1.979427e+08,4363.0,2.0,4.408346,2.222866,11,
8,NV,3546,489408.0,8.275679e+08,18186.0,9.0,4.395047,2.962657,12,
9,NH,1860,190937.0,3.170823e+08,6790.0,0.0,4.282800,2.471283,13,


saved -> /Users/ashvijain/Leading Indicator/output/tables/state_all.csv
saved -> /Users/ashvijain/Leading Indicator/output/tables/state_top10.csv


## Section 6 — Ranking by rate against ranking by count

Raw counts largely track how many people a sector employs. Dividing by hours worked gives a
rate that is comparable across sectors of different sizes. If the two rankings agreed, the
normalisation would add nothing. The comparison below is the point of the project.

In [26]:
# Top five by rate and top five by count, side by side, then the sectors that appear in one
# list and not the other, then the Spearman correlation between the two full rankings.
#
# Spearman rather than Pearson because both variables are ranks: it measures whether the
# order agrees, which is exactly the question. 1.0 would mean the two rankings are identical
# and normalising by hours changed nothing.

from scipy.stats import spearmanr

by_rate = sector.sort_values("rank_by_rate").reset_index(drop=True)
by_count = sector.sort_values("rank_by_count").reset_index(drop=True)

side_by_side = pd.DataFrame({
    "#": range(1, 6),
    "by rate": by_rate.sector.head(5).values,
    "trir": by_rate.trir.head(5).round(3).values,
    "by count": by_count.sector.head(5).values,
    "cases": by_count.total_cases.head(5).astype(int).values,
})
print("Top five sectors, each way:")
display(side_by_side)

top5_rate = set(by_rate.sector.head(5))
top5_count = set(by_count.sector.head(5))

lookup = sector.set_index("sector")[["rank_by_rate", "rank_by_count", "trir", "total_cases"]]

print("\nTop-5 by COUNT but not by RATE:")
display(lookup.loc[sorted(top5_count - top5_rate)].sort_values("rank_by_count"))

print("Top-5 by RATE but not by COUNT:")
display(lookup.loc[sorted(top5_rate - top5_count)].sort_values("rank_by_rate"))

print(f"In both top fives: {sorted(top5_rate & top5_count)}")

rho, pval = spearmanr(sector.rank_by_rate, sector.rank_by_count)
print(f"\nSpearman correlation between the two sector rankings: rho = {rho:.3f} "
      f"(p = {pval:.4f}, n = {len(sector)} sectors)")

# Largest rank movements in each direction, for the write-up.
# places_moved = rank_by_rate - rank_by_count, i.e. how far a sector moves when the ranking
# switches from raw counts to the hours-normalised rate. POSITIVE means it FALLS down the
# list once exposure is accounted for (it looked worse than it is on counts alone);
# NEGATIVE means it RISES (counts were hiding it).
moves = sector.assign(places_moved=sector.rank_by_rate - sector.rank_by_count)[
    ["sector", "rank_by_count", "rank_by_rate", "places_moved", "total_cases", "trir"]]
print("\nLargest rank movements (positive = falls when ranked by rate instead of count):")
display(moves.reindex(moves.places_moved.abs().sort_values(ascending=False).index)
             .head(6).reset_index(drop=True))

Top five sectors, each way:


,#,by rate,trir,by count,cases
0,1,Public Administration,5.615,Health Care and Social Assistance,333819
1,2,Transportation and Warehousing,5.451,Transportation and Warehousing,239852
2,3,"Arts, Entertainment, and Recreation",4.587,Manufacturing,222689
3,4,Retail Trade,4.502,Retail Trade,192744
4,5,Accommodation and Food Services,4.449,Construction,64756



Top-5 by COUNT but not by RATE:


,rank_by_rate,rank_by_count,trir,total_cases
sector,,,,
Health Care and Social Assistance,6,1,4.294043,333819.0
Manufacturing,12,3,2.541839,222689.0
Construction,15,5,1.565453,64756.0


Top-5 by RATE but not by COUNT:


,rank_by_rate,rank_by_count,trir,total_cases
sector,,,,
Public Administration,1,7,5.614515,53883.0
"Arts, Entertainment, and Recreation",3,10,4.587164,22768.0
Accommodation and Food Services,5,8,4.449023,40393.0


In both top fives: ['Retail Trade', 'Transportation and Warehousing']

Spearman correlation between the two sector rankings: rho = 0.716 (p = 0.0004, n = 20 sectors)

Largest rank movements (positive = falls when ranked by rate instead of count):


,sector,rank_by_count,rank_by_rate,places_moved,total_cases,trir
0,Construction,5,15,10,64756.0,1.565453
1,Manufacturing,3,12,9,222689.0,2.541839
2,"Arts, Entertainment, and Recreation",10,3,-7,22768.0,4.587164
3,Public Administration,7,1,-6,53883.0,5.614515
4,Real Estate and Rental and Leasing,15,9,-6,9973.0,3.256685
5,Health Care and Social Assistance,1,6,5,333819.0,4.294043


### The finding

**Ranking the 20 NAICS sectors by hours-normalised injury rate instead of by raw case count
replaces three of the top five: Health Care and Social Assistance, Manufacturing and
Construction drop out, and Public Administration, Arts, Entertainment and Recreation, and
Accommodation and Food Services take their places — with Construction falling the furthest,
from 5th by count (64,756 recordable cases) to 15th by rate (1.57 cases per 100 full-time
workers), below every sector except Professional Services, Information, Mining, Finance and
Management.**

The supporting values, all from the cell above:

- Only **Transportation and Warehousing** and **Retail Trade** hold a top-five place on both
  measures.
- **Health Care and Social Assistance** records the most recordable cases of any sector
  (333,819) but ranks 6th by rate (4.294). It has the largest case count because it works the
  second-largest number of hours (1.55 × 10¹⁰), not because its workplaces carry the highest
  rate.
- **Public Administration** ranks 1st by rate (5.615) and only 7th by count (53,883 cases).
- **Manufacturing** falls 9 places, from 3rd by count (222,689 cases) to 12th by rate (2.542).
- The two rankings agree only partly: **Spearman rho = 0.716** across the 20 sectors
  (p = 0.0004). Positive and clearly non-random, but far from the 1.0 that would mean
  normalising by hours had changed nothing.

The mechanism is exposure. Construction reports 8.27 × 10⁹ hours and Manufacturing
1.75 × 10¹⁰, so both accumulate large case counts from large workforces. Arts, Entertainment
and Recreation reports 9.93 × 10⁸ hours — about an eighteenth of Manufacturing's — and so
never appears near the top of a count ranking, despite a rate (4.587) nearly twice
Manufacturing's (2.542).

These are rates among **reporting establishments** in the OSHA ITA file, not national
industry rates; the sample is skewed toward large and high-hazard workplaces by OSHA's
reporting thresholds.

## Section 7 — Sanity check against a published rate

**Our overall TRIR on the clean table is 3.475** recordable cases per 100 full-time workers
(1,352,105 recordable cases over 77,817,253,795 hours, 369,996 establishments — computed in
Section 5, not typed in here). Deaths are excluded from that numerator and counted separately:
745.

**Most recent published BLS SOII total recordable case rate: [TBD]**
**Source URL: [TBD]**
*(Ashvi looks this up and pastes both in — PRD section 11, human task 3.)*

**Why the two will not match, and what would count as a failure.** This file covers only
establishments above OSHA's ITA reporting thresholds — 250+ employees outside the partially
exempt industries, plus 20–249-employee establishments in designated high-hazard industries —
so it is skewed toward large and high-hazard workplaces, while BLS SOII is a probability
sample designed to estimate the whole private-sector population. The ITA figure should
therefore run somewhat higher, and the same order of magnitude is the check that passes. A
gap of 10x in either direction would mean a bug in the pipeline, not a real difference between
the two populations.